## 摘要 Abstract

- 仅仅添加了六个空间视觉tokens来增强视觉表示。
  1. 我们提出了一种新型投影器，利用卷积核从ViT补丁特征中推导出视觉空间符号，模拟两种视觉空间排序方法：“从中心区域到全局”和“从抽象到具体”。然后，应用交叉注意力机制融合细粒度的视觉信息，丰富整体视觉表现。
  2. 我们提出了两个模型变体：LLaVA-SP-裁剪，通过渐进裁剪聚焦细节特征;以及 LLaVA-SP-池化，通过自适应池法捕捉全局语义，使模型能够处理多样化的视觉理解任务。
  3. 大量实验表明，LLaVA-SP 经过 LoRA 微调后，在多个多模态基准测试中实现显著性能提升，在多项任务中表现优于最先进的 LLaVA-1.5 模型，推理延迟几乎相同。

## 引言 Introduction

<!-- 多模态大语言模型（MLLMs）在理解和处理视觉与语言信息方面展现出卓越的能力，而跨模态理解的关键在于模态对齐。 -->
- 近期关于 MLLMs 中视觉与语言表征对齐的研究主要集中于视觉方面。为了减少 MLLMs 因视觉内容导致的**幻觉现象**，研究人员采用了多种策略，例如：
  - 提高图像分辨率、使用更强大的视觉编码器以及整合多种视觉特征。例如，LLaVA-1.5将输入图像分辨率提升至 336，
  - 而InternVL-1.5提出了一种动态高分辨率图像策略，支持 1024 分辨率图像输入。
  - SPHINX结合了多个视觉编码器以提取多样化的视觉特征。
  - Monkey并行地将不同图像块输入到各自的 ViT 编码器中以学习独特特征。
  - Mini-Gemini提出同时将低分辨率和高分辨率图像输入到视觉模型中。 
  - 然而，这些方法通常会导致视觉标记数量增加，从而显著增加训练和推理成本。


- 主流MLLM使用CLIP-ViT作为视觉编码器有两个局限:
  1. 对比学习范式在训练过程中依赖噪声较大的图像-文本对数据集，这限制了其理解细粒度感知细节的能力。
  2. ViT将二维图像分割成扁平的一维块序列，破坏相邻块之间的固有空间关系。研究表明，虽然ViT擅长捕捉全局信息，但在建模相邻patch块之间的局部关系存在困难。


<font color="yellow">
目标: 在不显著增加视觉tokens的前提下来增强视觉特征表示。
</font>

<!-- 为解决这一问题，我们提出了 LLaVA-SP 来增强 MLLMs 的视觉表征。LLaVA-SP 的投影器包含两个关键设计：空间特征提取器（SFE）和细节特征整合器（DFI）。
1) SFE 旨在通过添加仅六个视觉空间标记来增强视觉编码器的特征表征。这六个视觉空间标记可以通过裁剪或池化两种操作引入。裁剪的动机是强调详细区域特征，而池化则捕获图像的整体信息。在裁剪方法中，我们逐步向内裁剪 ViT 块特征，直到达到中心区域，从而获得多尺度特征。这些特征随后按照“从中心区域到全局”的顺序从左到右排列。裁剪侧重于区域细节，适合需要细粒度图像理解的任务。相比之下，池化方法使用自适应池化层生成多尺度特征，这些特征捕获不同抽象层次的信息，随后按照“从抽象到具体”的顺序从左到右排列。 这种策略受人类感知或创建图像的层次方式启发[52]，首先捕捉全局结构，然后聚焦局部细节。池化对于需要更广泛理解图像的任务尤其有益。对于这两种方法，ViT 块特征被重塑为其原始的 2D 形状。然后根据“从中心区域到全局”或“从抽象到具体”的策略重新组织，形成结构化的多尺度特征。最后，使用不同大小的卷积核对这些多尺度特征进行处理，以捕获视觉空间标记，这些标记与原始视觉标记连接，形成全面的视觉表示。
2) 2) DFI 通过交叉注意力机制进一步增强视觉空间特征。 在不增加 SFE 提取的视觉空间标记数量的情况下，DFI 从大型视觉特征图导出细粒度特征，并将它们整合到视觉空间标记中，以实现特征融合，这进一步增强了视觉表示，从而提高了 MLLMs 的细节理解能力。 -->


我们提出了 LLaVA-SP 来增强 MLLMs 的视觉表征。
LLaVA-SP 的投影器包含两个关键设计：空间特征提取器（Spatial Feature Extarctor, SFE）和细节特征整合器（Detail Feature Integrator, DFI）。
1) SFE 旨在通过添加仅六个视觉空间标记来增强视觉编码器的特征表征。这六个视觉空间标记可以通过裁剪(Cropping)或池化(Pooling)两种操作引入。
   * 裁剪的动机是强调详细区域特征，而池化则捕获图像的整体信息。在裁剪方法中，我们逐步向内裁剪 ViT 块特征，直到达到中心区域，从而获得多尺度特征。这些特征随后按照“从中心区域到全局”的顺序从左到右排列。裁剪侧重于区域细节，适合需要细粒度图像理解的任务。
   * 相比之下，池化方法使用自适应池化层生成多尺度特征，这些特征捕获不同抽象层次的信息，随后按照“从抽象到具体”的顺序从左到右排列。 这种策略受人类感知或创建图像的层次方式启发，首先捕捉全局结构，然后聚焦局部细节。池化对于需要更广泛理解图像的任务尤其有益。对于这两种方法，ViT 块特征被重塑为其原始的 2D 形状。然后根据“从中心区域到全局”或“从抽象到具体”的策略重新组织，形成结构化的多尺度特征。最后，使用不同大小的卷积核对这些多尺度特征进行处理，以捕获视觉空间标记，这些标记与原始视觉标记连接，形成全面的视觉表示。
2) DFI 通过交叉注意力机制进一步增强视觉空间特征。 在不增加 SFE 提取的视觉空间标记数量的情况下，DFI 从大型视觉特征图导出细粒度特征，并将它们整合到视觉空间标记中，以实现特征融合，这进一步增强了视觉表示，从而提高了 MLLMs 的细节理解能力。

我们的主要贡献包括：

<!-- - Visual spatial tokens enhance the visual representation of MLLMs. We propose a novel Projector to capture visual spatial tokens, effectively extracting the spatial information among local adjacent ViT patch features. -->
<!-- - Two model variants handle diverse tasks. LLaVA-SP-Cropping focuses on detailed features, while LLaVA-SP-Pooling captures global semantics, handling fine-grained and general visual understanding tasks respectively. -->
<!-- - Performance improvements on various multimodal benchmarks. Fig. 1 demonstrates that LLaVA-SP finetuned with LoRA [20] outperform LLaVA-1.5 on various multimodal benchmarks. -->
- **视觉空间token增强了 MLLMs 的视觉表征**。我们提出了一种新的投影器来捕获视觉空间标记，有效地提取局部相邻 ViT 块特征之间的空间信息。
- **两种模型变体处理不同任务**。LLaVA-SP-Cropping 专注于细节特征，而 LLaVA-SP-Pooling 捕获全局语义，分别处理细粒度和通用视觉理解任务。
- **在各种多模态基准测试上的性能提升**。图 [1](#ladar_eval) 表明，使用 LoRA 微调的 LLaVA-SP 在各种多模态基准测试上优于 LLaVA-1.5。

<a id="ladar_eval"></a>
<div style="background-color:#f9f9f9; padding:10px; width:60%; margin:auto; text-align:center;">
    <image src="./assets/ladar_eval.png"/>
    <span style="font-size:12px; color:#555;">图1. LLaVA-SP在LoRA微调后，11个多模态基准测试中10个优于完全训练的LLaVA-1.5</span>
</div>

## 方法 Methods

<div style="background-color:#f9f9f9; padding:10px; width:60%; margin:auto; text-align:center;">
    <image src="./assets/achichect.png"/>
    <span style="font-size:12px; color:#555;">LLaVA-SP 的架构基于 LLaVA-1.5的结构。投影器具有两个并行分支，左分支专门用于提取视觉空间标记。</span>
</div>

### 概述 Overview

__投影器 Projector__

投影器。投影器将视觉特征映射到大型语言模型的语言表示空间中。它由三个组件组成：SFE（可训练卷积矩阵 $W_c$ ）、DFI（可训练线性矩阵 $W_d$ ）和两个并行的 MLP（ $W_s$ 和 $W_p$ ）。SFE 通过从 ViT 块特征 $Z_p$ 中提取视觉空间特征 $Z_s$ 来开始这个过程。DFI 通过整合小尺度（ $Z_{s⁢-⁢\text{small}}$ ）和大规模（ $Z_{s⁢-⁢\text{b⁢i⁢g}}$ ）特征来挖掘细粒度特征，进一步丰富视觉空间特征的细节 $Z_{v⁢s}$ 。两个并行的 MLP 执行专业转换： $W_s$ 将空间特征 $Z_{v⁢s}$ 转换为视觉空间标记 $H_{v⁢s}$ ，而 $W_p$ 将 ViT 块特征 $Z_p$ 转换为视觉块标记 $H_{v⁢p}$ 。这种双重映射确保不同的视觉特征被独立处理，保留个性化信息，并在一致表示空间中对齐。 


### 空间特征提取器 Spatial Feature Extractor (SFE)

- 传统的视觉tokens -> 1D序列破坏了视觉特征的2D空间关系。
- 我们提出SFE，作为原始视觉信息表示的补充，遵循2个原则：
  1. 获取能够捕获图像2D空间结构的多尺度特征。
  2. 使用卷积核提取视觉空间特征。

为了获得多尺度特征，可以对ViT块特征进行剪裁(Cropping)或池化(Pooling)操作。


__裁剪 Cropping:__ 图 3(a)显示通过裁剪获得多尺度特征。SFE 将 CLIP-ViT-L/14-336 patch 特征重新排列成其原始的 2D 形状 $Z_p \in \mathbb{R}^{\sqrt N \times \sqrt N \times C}$ ，其中 $N = 576$ 表示视觉块的数量， C 表示特征维度。在第一步中，我们获得所有 ViT patch 特征 $Z_{p6} = Z_p \in \mathbb{R}^{24 \times 24 \times C}$ 。在第二步中，使用 $Z_{p⁢6}$ 作为参考，以步长=2 向内裁剪以获得 $Z_{p5} \in  \mathbb{R}^{20 \times 20 \times C}$ 。这个特征裁剪过程重复进行，直到剩余的中心区域特征太小而无法裁剪，如图 3(a)中的 $Z_{p⁢1}\in \mathbb{R}^{4 \times 4 \times C}$ 。这个过程生成了按“从中心区域到全局”排列的多尺度特征 $(Z_{p⁢1},Z_{p⁢2},Z_{p⁢3},Z_{p⁢4},Z_{p⁢5},Z_{p⁢6})$ ，强调图像区域中的细节。

__池化 Pooling:__ 图 3(b)显示了通过池化获取多尺度特征。SFE 使用自适应平均池化来模拟从抽象到具体的视觉感知和创建过程。较小的特征图丢失更多信息，代表更抽象的信息，而较大的特征图传递更多具体细节。多尺度特征序列按“从抽象到具体”排列，强调图像全局语义。

<div style="background-color:#f9f9f9; padding:10px; width:80%; margin:auto; text-align:center;">
    <div style="display:flex">
        <div>
            <image src="./assets/crop.png" style="margin-top:15px"/>
            <span style="font-size:12px; color:#555;">(a) LLaVA-SP-Cropping模型中的SFE模块</span>
        </div>
        <div>
            <image src="./assets/pooling.png"/>
            <span style="font-size:12px; color:#555;">(b) LLaVA-SP-Pooling模型中的SFE模块</span>
        </div>
    </div>
    <span style="font-size:12px; color:#555; margin-top:10px;">图2. SFE结构</span>
</div>

接下来，我们利用卷积的固有空间建模能力来提取空间特征。尺寸为$k= 4, 8, 12, 16, 20, 24$的卷积核可以完全覆盖($Z_{p⁢1},Z_{p⁢2},Z_{p⁢3},Z_{p⁢4},Z_{p⁢5},Z_{p⁢6}$),并通过在序列维度上一次连接来计算视觉空间特征$Z_{s-\text{small}}$:
$$\begin{aligned}
Z_{si} &= \text{conv}_k (Z_{pi; k = 4i, i = 1, 2, 3, 4, 5, 6}) \\
Z_{s-\text{small}} &= \text{concat}(Z_{s1}, Z_{s2}, Z_{s3}, Z_{s4}, Z_{s5}, Z_{s6}) \\
\end{aligned}$$

- 其中$Z_{si} \in \mathbb{R}^{1 \times 1 \times C}$
- $Z_{s-\text{small}} \in \mathbb{R}^{6 \times 1 \times C}$

<div style="background-color:#f9f9f9; padding:10px; width:60%; margin:auto; text-align:center;">
    <image src="./assets/fuse.png"/>
    <span style="font-size:12px; color:#555;">图4: DFI架构，整合$Z_{s-\text{big}}$并将其注入$Z_{s-\text{small}}$</span>
</div>

![表 1](assets/IMG_1125-195759333.png)  


### 细节特征整合器 Detail Feature Integrator (DFI)


<!-- Our goal in designing DFI was to address the trade-off in SFE, where large convolution kernels capture a broad receptive field but miss finer details, while smaller kernels increase token count. To avoid increasing visual spatial tokens, and thus prevent the training and inference costs associated with long input sequences to LLM. DFI uses an attention mechanism to inject fine-grained features from smaller convolution kernels into the six tokens generated by SFE.
我们在设计 DFI 时的目标是解决 SFE 中的权衡问题，即大卷积核能够捕获广阔的感受野但会丢失更精细的细节，而小卷积核会增加 token 数量。为了避免增加视觉空间 token，从而防止与 LLM 长输入序列相关的训练和推理成本，DFI 使用注意力机制将来自较小卷积核的细粒度特征注入由 SFE 生成的六个 token 中。
Mentioned in  Sec. 3.2, Zs⁢-⁢s⁢m⁢a⁢l⁢l represents six visual spatial features. Zs⁢-⁢b⁢i⁢g is a feature map extracted using smaller kernels (the deep blue kernel on the far right of Conv Group in Figs. 3(a) and 3(b)). As shown in Fig. 4: Zs⁢-⁢s⁢m⁢a⁢l⁢l is used as the query, while Zs⁢-⁢b⁢i⁢g serves as the key and value. Through the cross-attention mechanism, fine-grained features are mined from Zs⁢-⁢b⁢i⁢g and injected into Zs⁢-⁢s⁢m⁢a⁢l⁢l. Then we c⁢o⁢n⁢c⁢a⁢t attention features and Zs⁢-⁢s⁢m⁢a⁢l⁢l in channel dimension, extracting visual spatial features Zv⁢s:
如第 3.2 节所述， Zs⁢-⁢s⁢m⁢a⁢l⁢l 代表六个视觉空间特征。 Zs⁢-⁢b⁢i⁢g 是使用较小卷积核（在 Figs. 3(a)和 3(b)中 Conv Group 最右边的深蓝色卷积核）提取的特征图。如图 4 所示： Zs⁢-⁢s⁢m⁢a⁢l⁢l 用作查询，而 Zs⁢-⁢b⁢i⁢g 作为键和值。通过交叉注意力机制，从 Zs⁢-⁢b⁢i⁢g 中挖掘细粒度特征并注入到 Zs⁢-⁢s⁢m⁢a⁢l⁢l 中。然后我们 c⁢o⁢n⁢c⁢a⁢t 注意力特征并在通道维度上 Zs⁢-⁢s⁢m⁢a⁢l⁢l ，提取视觉空间特征 Zv⁢s ： -->

$$\begin{aligned}
Z_{vs} = \text{concat}([Z_{s-\text{small}}, softmax(\frac{Q\times K^T}{\sqrt{d_k}})V])
\end{aligned}$$

其中：
- $Z_{vs} \in \mathbb{R}^{6 \times 1 \times 2C}$
- $d_k$ 是特征维度
- $Q = W_Q(Z_{s-\text{small}})$
- $K = W_K(Z_{s-\text{big}})$
- $V = W_V(Z_{s-\text{big}})$